<a href="https://colab.research.google.com/github/CodeByQasim/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CodeByQasim/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## Lane 3 — Structured Content Archetype Clustering

This notebook builds a simple, explainable baseline action score for content review.

The baseline uses two observable signals:

1. **Freshness / staleness** — how many days have passed since the last content update.
2. **Visibility / demand** — measured using impressions over the last 90 days.

The goal is not to predict Google rankings or make causal claims. The baseline is only a decision-support queue that identifies pages that are both relatively stale and still visible enough to deserve review.

The Week 5 model should provide evidence that it can improve on this simple baseline.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Solution:

### Signal checks

Before creating the baseline rule, I checked two signals:

* **Staleness:** overall relationship with the observed trend signal is mixed, so staleness alone is not enough to prioritize a page.
* **Visibility / impressions:** higher-impression pages generally have more observed clicks and sessions, so visibility is useful for deciding which stale pages deserve attention.

### Verdicts

* Staleness / refresh-flag-linked signal: **MIXED**
* Visibility / volume signal: **CONFIRMED**

### Baseline rule

I will prioritize a page when:

> `days_since_last_update >= 180` AND `impressions_90d >= 500`

This means the page has not been updated for at least 180 days and has enough observed visibility to justify review.

### Reason code

`stale_visible_page`

### Action label

`refresh_review`

Pages that do not meet this rule receive:

* Reason code: `general_review`
* Action: `monitor`

This is intentionally a simple one-rule baseline so that the Week 5 model has a clear benchmark to beat.


In [9]:
from google.colab import drive
import pandas as pd

# Mount Google Drive
drive.mount('/content/drive')

# Dataset path
file_path = "/content/drive/MyDrive/Flyrank Dataset/content_refresh_anonymized.csv"

# Load dataset
df = pd.read_csv(file_path)

# Check dataset
print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

display(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## Signal 1 — Staleness Check

I will inspect the distribution of pages across freshness/staleness buckets.

The purpose is to check whether older pages consistently show worse observed performance.

If the relationship is not monotonic, I will treat staleness as a useful condition rather than claiming that age alone causes poor performance.


In [10]:
import numpy as np
import pandas as pd

# Create staleness buckets
staleness_bins = [-1, 29, 89, 179, np.inf]
staleness_labels = [
    "0-29 days",
    "30-89 days",
    "90-179 days",
    "180+ days"
]

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=staleness_bins,
    labels=staleness_labels
)

# Create observed trend indicator for audit only
trend_down = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
)

# Create summary table
staleness_table = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "count"),
          median_impressions=("impressions_90d", "median"),
          median_clicks=("clicks_90d", "median"),
          median_position=("avg_position", "median")
      )
      .reset_index()
)

# Add observed down rate
staleness_table["observed_down_rate"] = (
    df.groupby("staleness_bucket", observed=False)["trend_direction"]
      .apply(
          lambda x: x.astype(str)
                    .str.lower()
                    .eq("down")
                    .mean()
      )
      .values
)

display(staleness_table)

# Check stale + visible rule
stale_visible = (
    (df["days_since_last_update"] >= 180) &
    (df["impressions_90d"] >= 500)
)

print("\nStale + visible pages:")
print("n =", stale_visible.sum())
print("share =", round(stale_visible.mean() * 100, 3), "%")

print(
    "Median impressions among stale + visible pages =",
    df.loc[stale_visible, "impressions_90d"].median()
)

print(
    "Median clicks among stale + visible pages =",
    df.loc[stale_visible, "clicks_90d"].median()
)

print(
    "Median position among stale + visible pages =",
    df.loc[stale_visible, "avg_position"].median()
)

print("\nVerdict: MIXED")

,staleness_bucket,n,median_impressions,median_clicks,median_position,observed_down_rate
0,0-29 days,20480,470.0,1.0,9.9,0.511377
1,30-89 days,175,510.0,0.0,13.9,0.588571
2,90-179 days,9171,1692.0,2.0,13.6,0.611057
3,180+ days,174,15.5,0.0,7.0,0.471264



Stale + visible pages:
n = 17
share = 0.057 %
Median impressions among stale + visible pages = 4429.0
Median clicks among stale + visible pages = 4.0
Median position among stale + visible pages = 18.6

Verdict: MIXED


## Signal 2 — Visibility / Volume Check

I will bucket pages according to their observed 90-day impressions.

The purpose is to check whether pages with greater visibility also have enough observed activity to justify prioritization.

Impressions are used as a visibility signal, not as a causal explanation of performance.

**Verdict: CONFIRMED**


In [11]:
# Create impression buckets
volume_bins = [-1, 99, 499, 4999, np.inf]
volume_labels = [
    "0-99",
    "100-499",
    "500-4,999",
    "5,000+"
]

df["volume_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=volume_bins,
    labels=volume_labels
)

# Create summary table
volume_table = (
    df.groupby("volume_bucket", observed=False)
      .agg(
          n=("content_id", "count"),
          median_clicks=("clicks_90d", "median"),
          median_sessions=("sessions_90d", "median"),
          median_position=("avg_position", "median")
      )
      .reset_index()
)

# Observed trend information for audit context only
volume_table["observed_down_rate"] = (
    df.groupby("volume_bucket", observed=False)["trend_direction"]
      .apply(
          lambda x: x.astype(str)
                    .str.lower()
                    .eq("down")
                    .mean()
      )
      .values
)

display(volume_table)

print("\nVerdict: CONFIRMED")

print(
    "Higher-impression buckets generally show higher observed "
    "clicks and sessions, supporting impressions as a useful "
    "visibility/priority signal."
)

,volume_bucket,n,median_clicks,median_sessions,median_position,observed_down_rate
0,0-99,7994,0.0,2.0,7.5,0.389042
1,100-499,5280,0.0,4.0,16.2,0.604356
2,"500-4,999",10575,2.0,11.0,12.9,0.623924
3,"5,000+",6151,29.0,65.0,8.3,0.546740



Verdict: CONFIRMED
Higher-impression buckets generally show higher observed clicks and sessions, supporting impressions as a useful visibility/priority signal.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## Build the Ranked Queue

The baseline uses one simple rule:

> stale AND visible

The score combines two observable signals:

* 60% freshness/staleness
* 40% visibility

Only pages satisfying the baseline rule receive a positive action score.

The score is a ranking score, not a probability and not a prediction of future performance.

The score does not use `trend_direction`. That field is used only for descriptive signal auditing.


In [12]:
# Make a copy for the ranking queue
queue = df.copy()

# ---------------------------------------------------------
# 1. Visibility score
# ---------------------------------------------------------
# log1p reduces the influence of extremely large impression values
log_impressions = np.log1p(queue["impressions_90d"])

visibility_score = log_impressions.rank(pct=True)

# ---------------------------------------------------------
# 2. Freshness / staleness score
# ---------------------------------------------------------
freshness_score = queue["days_since_last_update"].rank(pct=True)

# ---------------------------------------------------------
# 3. Baseline rule
# ---------------------------------------------------------
queue["rule_hit"] = (
    (queue["days_since_last_update"] >= 180) &
    (queue["impressions_90d"] >= 500)
)

# ---------------------------------------------------------
# 4. Combined score
# ---------------------------------------------------------
combined_score = (
    0.60 * freshness_score +
    0.40 * visibility_score
)

queue["baseline_action_score"] = np.where(
    queue["rule_hit"],
    50 + 50 * combined_score,
    0
)

# ---------------------------------------------------------
# 5. Reason code
# ---------------------------------------------------------
queue["reason_code"] = np.where(
    queue["rule_hit"],
    "stale_visible_page",
    "general_review"
)

# ---------------------------------------------------------
# 6. Action
# ---------------------------------------------------------
queue["action"] = np.where(
    queue["rule_hit"],
    "refresh_review",
    "monitor"
)

# ---------------------------------------------------------
# 7. Rank all rows
# ---------------------------------------------------------
queue = queue.sort_values(
    ["baseline_action_score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

# ---------------------------------------------------------
# 8. Select output columns
# ---------------------------------------------------------
output_columns = [
    "content_id",
    "rank",
    "baseline_action_score",
    "reason_code",
    "action",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "word_count"
]

output_columns = [
    col for col in output_columns
    if col in queue.columns
]

baseline_output = queue[output_columns].copy()

# ---------------------------------------------------------
# 9. Save output
# ---------------------------------------------------------
output_dir = "work/outputs"

import os
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "baseline_action_score.csv"
)

baseline_output.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Rows:", len(baseline_output))

display(baseline_output.head(10))


Saved: work/outputs/baseline_action_score.csv
Rows: 30000


,content_id,rank,baseline_action_score,reason_code,action,impressions_90d,clicks_90d,sessions_90d,avg_position,content_age_days,days_since_last_update,word_count
0,content_cf56e2e2e282,1,99.613000,stale_visible_page,refresh_review,61678,94,119,19.7,231,194,5125.0
1,content_7368877ea310,2,99.605000,stale_visible_page,refresh_review,59472,77,82,24.8,231,194,2591.0
2,content_1bfaa38ff26c,3,98.993000,stale_visible_page,refresh_review,25715,60,80,22.2,231,194,3861.0
3,content_0a91db491d14,4,98.037000,stale_visible_page,refresh_review,13299,65,78,10.5,231,193,3478.0
4,content_5feee3994adb,5,96.937667,stale_visible_page,refresh_review,7812,1,5,39.0,231,194,3590.0
5,content_c2d929d83eaa,6,96.852333,stale_visible_page,refresh_review,7558,15,25,17.9,231,193,4758.0
6,content_b16bd7307b39,7,95.560000,stale_visible_page,refresh_review,4590,0,4,31.0,231,194,4329.0
7,content_fe16a55cd13d,8,95.538000,stale_visible_page,refresh_review,4556,15,42,16.4,231,194,3388.0
8,content_ecb6215e79fd,9,95.450000,stale_visible_page,refresh_review,4429,17,12,25.3,231,194,4486.0
9,content_928af3e22c80,10,92.530667,stale_visible_page,refresh_review,1697,2,3,15.8,231,193,3118.0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

##  Top-20 Review

I will review the 20 highest-ranked rows.

For each row, I will report:

* the proposed action,
* the reason code,
* a confidence note,
* and what could make the recommendation wrong.

Confidence is based only on observed data and should not be interpreted as certainty.


In [13]:
# Select the top 20 rows
top20 = queue.head(20).copy()

# ---------------------------------------------------------
# Confidence note
# ---------------------------------------------------------
def confidence_note(row):

    if row["action"] == "monitor":
        return "Low — row did not meet the baseline rule."

    if row["impressions_90d"] >= 5000:
        return "High — strong observed visibility."

    elif row["impressions_90d"] >= 1000:
        return "Medium-high — meaningful observed visibility."

    else:
        return "Medium — rule hit, but visibility is more limited."


# ---------------------------------------------------------
# What could make the recommendation wrong?
# ---------------------------------------------------------
def wrong_reason(row):

    if row["action"] == "monitor":
        return (
            "The page may still need review for reasons not "
            "captured by this one-rule baseline."
        )

    if row["impressions_90d"] < 1000:
        return (
            "Low visibility may mean the refresh opportunity "
            "is smaller than the score suggests."
        )

    if row["avg_position"] > 20:
        return (
            "Weak observed position may indicate limited "
            "opportunity or another underlying issue."
        )

    return (
        "Seasonality, demand changes, consolidation, or other "
        "page-level factors could make the rule inappropriate."
    )


top20["confidence_note"] = top20.apply(
    confidence_note,
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    wrong_reason,
    axis=1
)

# ---------------------------------------------------------
# Display review table
# ---------------------------------------------------------
review_columns = [
    "rank",
    "content_id",
    "action",
    "reason_code",
    "baseline_action_score",
    "impressions_90d",
    "clicks_90d",
    "avg_position",
    "confidence_note",
    "what_would_make_it_wrong"
]

top20_review = top20[review_columns]

display(top20_review)

,rank,content_id,action,reason_code,baseline_action_score,impressions_90d,clicks_90d,avg_position,confidence_note,what_would_make_it_wrong
0,1,content_cf56e2e2e282,refresh_review,stale_visible_page,99.613000,61678,94,19.7,High — strong observed visibility.,"Seasonality, demand changes, consolidation, or..."
1,2,content_7368877ea310,refresh_review,stale_visible_page,99.605000,59472,77,24.8,High — strong observed visibility.,Weak observed position may indicate limited op...
2,3,content_1bfaa38ff26c,refresh_review,stale_visible_page,98.993000,25715,60,22.2,High — strong observed visibility.,Weak observed position may indicate limited op...
3,4,content_0a91db491d14,refresh_review,stale_visible_page,98.037000,13299,65,10.5,High — strong observed visibility.,"Seasonality, demand changes, consolidation, or..."
4,5,content_5feee3994adb,refresh_review,stale_visible_page,96.937667,7812,1,39.0,High — strong observed visibility.,Weak observed position may indicate limited op...
5,6,content_c2d929d83eaa,refresh_review,stale_visible_page,96.852333,7558,15,17.9,High — strong observed visibility.,"Seasonality, demand changes, consolidation, or..."
6,7,content_b16bd7307b39,refresh_review,stale_visible_page,95.560000,4590,0,31.0,Medium-high — meaningful observed visibility.,Weak observed position may indicate limited op...
7,8,content_fe16a55cd13d,refresh_review,stale_visible_page,95.538000,4556,15,16.4,Medium-high — meaningful observed visibility.,"Seasonality, demand changes, consolidation, or..."
8,9,content_ecb6215e79fd,refresh_review,stale_visible_page,95.450000,4429,17,25.3,Medium-high — meaningful observed visibility.,Weak observed position may indicate limited op...
9,10,content_928af3e22c80,refresh_review,stale_visible_page,92.530667,1697,2,15.8,Medium-high — meaningful observed visibility.,"Seasonality, demand changes, consolidation, or..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

##  Weak Picks + Leakage Check

### Weak picks

Because this baseline uses one strict rule, only pages satisfying the rule receive a positive score.

If fewer than 20 pages satisfy the rule, the remaining Top-20 rows can have a score of zero. These rows should not be interpreted as strong recommendations; they are included so that the fixed Top-20 queue can still be inspected.

### Leakage check

The baseline must not use product decision flags or future-window information.

The actual score inputs are only:

* `days_since_last_update`
* `impressions_90d`

`trend_direction` is used only for descriptive auditing and is not used to calculate the score.


In [14]:
# ---------------------------------------------------------
# 1. Weak picks
# ---------------------------------------------------------
weak_picks = top20[
    top20["baseline_action_score"] == 0
][[
    "rank",
    "content_id",
    "baseline_action_score",
    "reason_code",
    "action",
    "impressions_90d",
    "avg_position"
]]

print("Weak picks in Top-20:")

display(weak_picks)


# ---------------------------------------------------------
# 2. Check product decision flags
# ---------------------------------------------------------
forbidden_product_flags = [
    "health_score",
    "priority_score",
    "action_type",
    "refresh_tier",
    "needs_ctr_fix",
    "is_quick_win"
]

present_product_flags = [
    col
    for col in forbidden_product_flags
    if col in queue.columns
]

print("\nProduct decision flag columns present:")
print(present_product_flags)

if len(present_product_flags) == 0:
    print("PASS: No product decision flags are present.")
else:
    print("WARNING: Product decision flag columns are present.")


# ---------------------------------------------------------
# 3. Check future-window fields
# ---------------------------------------------------------
future_keywords = [
    "next_30d",
    "next_28d",
    "future_clicks",
    "future_sessions",
    "future_impressions",
    "target_window"
]

future_columns = [
    col
    for col in queue.columns
    if any(
        keyword in col.lower()
        for keyword in future_keywords
    )
]

print("\nFuture-window columns detected:")
print(future_columns)

if len(future_columns) == 0:
    print("PASS: No obvious future-window columns detected.")
else:
    print("WARNING: Future-window columns detected.")


# ---------------------------------------------------------
# 4. Show actual score inputs
# ---------------------------------------------------------
score_inputs = [
    "days_since_last_update",
    "impressions_90d"
]

print("\nActual baseline score inputs:")
print(score_inputs)

print("\nLEAKAGE CHECK: PASS")

Weak picks in Top-20:


,rank,content_id,baseline_action_score,reason_code,action,impressions_90d,avg_position
17,18,content_5fe46e04994d,0.0,general_review,monitor,517715,4.2
18,19,content_aaef01a50def,0.0,general_review,monitor,517109,5.4
19,20,content_8c19996aa890,0.0,general_review,monitor,509252,2.5



Product decision flag columns present:
[]
PASS: No product decision flags are present.

Future-window columns detected:
[]
PASS: No obvious future-window columns detected.

Actual baseline score inputs:
['days_since_last_update', 'impressions_90d']

LEAKAGE CHECK: PASS


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.